# Route definition 

In [2]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Configuración de rutas (asegúrate de que ROOT suba a la raíz)
ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
from config.paths import RAW_DIR, PROCESSED_DIR

YEARS = [2018, 2024]


# Imputation of IVA

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

# Configuración de prefijos IVA y códigos de informalidad por año
YEAR_CONFIG = {
    2018: {
        # Prefijos: C (Limpieza), D (Cuidados), F (Comunicaciones/Combustible), 
        # H (Vestido), I (Muebles), K (Enseres), L (Esparcimiento), M (Transporte), N (Otros)
        "iva_prefixes": ('C', 'D', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N'),
        "informal_locs": [1, 2, 3, 17], # Códigos 2018
        "clave_len": 4
    },
    2024: {
        # Prefijos COICOP: 02 (Alcohol), 03 (Vestido), 05 (Muebles), 07 (Transporte), 
        # 08 (Comunicación), 09 (Recreación), 11 (Restaurantes), 12 (Otros)
        "iva_prefixes": ('012', '02', '03', '05', '07', '08', '09', '11', '12'),
        "informal_locs": [1, 2, 3, 17], # Generalmente se mantienen, verificar en FD
        "clave_len": 6
    }
}

def fix_and_calculate_iva(year, raw_dir):
    print(f"🛠️  Procesando ENIGH {year}...")
    
    config = YEAR_CONFIG[year]
    
    # 1. Carga de datos
    df_gastos = pd.read_csv(raw_dir / f"gastoshogar_{year}.csv", 
                            dtype={'folioviv': str, 'foliohog': str, 'clave': str}, 
                            low_memory=False)
    df_conc = pd.read_csv(raw_dir / f"concentradohogar_{year}.csv", 
                          dtype={'folioviv': str, 'foliohog': str}, 
                          low_memory=False)

    # 2. Procesamiento de Gastos
    df_gastos['gasto_tri'] = pd.to_numeric(df_gastos['gasto_tri'], errors='coerce').fillna(0)
    
    # Filtrar solo gastos monetarios (G1)
    df_gastos = df_gastos[df_gastos['tipo_gasto'] == 'G1'].copy()
    
    # Identificar si lleva IVA basado en los prefijos del año correspondiente
    df_gastos['lleva_iva'] = df_gastos['clave'].str.startswith(config["iva_prefixes"]).astype(int)
    
    # Identificar formalidad 
    df_gastos['es_formal'] = (~df_gastos['lugar_comp'].isin(config["informal_locs"])).astype(int)
    
    # Cálculo del IVA por partida: Gasto * (0.16 / 1.16)
    # $Factor_{IVA} = \frac{0.16}{1.16} \approx 0.137931$ 
    factor_iva = 0.137931034
    df_gastos['iva_partida'] = np.where((df_gastos['lleva_iva'] == 1) & (df_gastos['es_formal'] == 1),
                                        df_gastos['gasto_tri'] * factor_iva, 0)
    
    # Agrupar IVA por hogar
    iva_hogar = df_gastos.groupby(['folioviv', 'foliohog'])['iva_partida'].sum().reset_index()
    iva_hogar.rename(columns={'iva_partida': 'IVA_HH'}, inplace=True)

    # 3. Preparar Concentrado
    cols_num = ['gasto_mon', 'alimentos', 'limpieza', 'cuidados', 'ing_cor', 'factor']
    for col in cols_num:
        df_conc[col] = pd.to_numeric(df_conc[col], errors='coerce').fillna(0)
    
    # Base de gasto estructural (excluyendo lo básico que suele ser tasa 0% o exento)
    df_conc['Gasto_HH_tri'] = df_conc['gasto_mon'] - (df_conc['alimentos'] + df_conc['limpieza'] + df_conc['cuidados'])

    # 4. Merge y cálculo de tasa efectiva
    df_final = pd.merge(df_conc, iva_hogar, on=['folioviv', 'foliohog'], how='left').fillna(0)
    
    # Cálculo del IVA atribuible al ingreso (proporcional)
    # Evitamos división por cero con np.where
    df_final['IVA_HH_A'] = np.where(df_final['Gasto_HH_tri'] > 0,
                                    (df_final['IVA_HH'] / df_final['Gasto_HH_tri']) * df_final['ing_cor'],
                                    0)
    
    return df_final.copy()

# Ejecución del Loop
results = {}
for year in [2018, 2024]:
    df_result = fix_and_calculate_iva(year, RAW_DIR)
    
    # Cálculo de Deciles
    df_result = df_result.sort_values("ing_cor")
    df_result["cumsum_factor"] = df_result["factor"].cumsum()
    df_result["decil"] = pd.qcut(df_result["cumsum_factor"], 10, labels=range(1, 11))
    
    # Resumen ponderado
    summary = df_result.groupby("decil").agg(
        ingreso_medio=("ing_cor", lambda x: np.average(x, weights=df_result.loc[x.index, "factor"])),
        iva_medio=("IVA_HH_A", lambda x: np.average(x, weights=df_result.loc[x.index, "factor"])),
        hogares_representados=("factor", "sum")
    ).reset_index()
    
    summary["carga_iva_pct"] = (summary["iva_medio"] / summary["ingreso_medio"]) * 100
    results[year] = summary

    print(f"\n{'='*20} ENIGH {year} {'='*20}")
    display(summary.style.format({
        "ingreso_medio": "${:,.2f}", "iva_medio": "${:,.2f}", 
        "carga_iva_pct": "{:.2f}%", "hogares_representados": "{:,.0f}"
    }))

🛠️  Procesando ENIGH 2018...

==================== ENIGH 2018 ====================


/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_2943/3005975830.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_conc['Gasto_HH_tri'] = df_conc['gasto_mon'] - (df_conc['alimentos'] + df_conc['limpieza'] + df_conc['cuidados'])
/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_2943/3005975830.py:71: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['IVA_HH_A'] = np.where(df_final['Gasto_HH_tri'] > 0,


,decil,ingreso_medio,iva_medio,hogares_representados,carga_iva_pct
0,1,"$8,716.22","$1,658.35","3,077,839",19.03%
1,2,"$15,291.60","$2,251.26","3,127,662",14.72%
2,3,"$20,324.76","$2,223.45","3,221,401",10.94%
3,4,"$25,314.82","$2,806.68","3,299,920",11.09%
4,5,"$30,614.86","$3,098.35","3,412,828",10.12%
5,6,"$36,961.89","$3,629.89","3,460,309",9.82%
6,7,"$44,727.99","$4,388.86","3,501,973",9.81%
7,8,"$55,585.77","$5,446.05","3,575,937",9.80%
8,9,"$73,631.94","$8,816.83","3,682,880",11.97%
9,10,"$156,510.85","$18,790.49","4,039,766",12.01%


🛠️  Procesando ENIGH 2024...

==================== ENIGH 2024 ====================


/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_2943/3005975830.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_conc['Gasto_HH_tri'] = df_conc['gasto_mon'] - (df_conc['alimentos'] + df_conc['limpieza'] + df_conc['cuidados'])
/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_2943/3005975830.py:71: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['IVA_HH_A'] = np.where(df_final['Gasto_HH_tri'] > 0,


,decil,ingreso_medio,iva_medio,hogares_representados,carga_iva_pct
0,1,"$15,688.53","$5,014.86","3,283,413",31.97%
1,2,"$26,482.16","$3,675.52","3,532,617",13.88%
2,3,"$34,539.90","$4,416.71","3,634,153",12.79%
3,4,"$42,452.03","$5,475.26","3,746,813",12.90%
4,5,"$51,059.62","$6,283.88","3,861,828",12.31%
5,6,"$60,895.15","$7,609.87","3,949,553",12.50%
6,7,"$72,846.22","$9,077.25","3,922,847",12.46%
7,8,"$89,292.18","$12,463.10","4,055,830",13.96%
8,9,"$115,561.16","$14,573.56","4,229,375",12.61%
9,10,"$220,959.36","$34,181.70","4,613,801",15.47%
